# Step 18 — Where the class bias lives, when trajectories commit, and making the CIFAR census classifier-independent

Self-contained. GPU on, Internet on. ~4 h as a committed run (MNIST parts ~45 min; CIFAR classifier + census + floor ~3 h).

**A. Band-limited drift replacement** (4 pairs × 3 seeds, epoch 20). The network's projected drift is replaced by the exact one in one noise band at a time — high noise ($\tau>1.5$), the window ($0.3\le\tau\le1.5$), low noise ($\tau<0.3$) — and over all $\sigma$. Which band carries the class bias? Heun-40 and SDE-40, CNN labels, mean $\pm$ sd over seeds.

**B. Slow passage.** For sampled trajectories, the noise level at which the denoised LDA coordinate last changes sign (commitment), compared with the exact and network $\tau_{\rm bif}$; for the trained models and for an exact-score $d=784$ mixture.

**C. CIFAR-10**: (i) a symmetrised-control floor along the classifier log-odds; (ii) classifier blindness: each pair's accuracy-vs-$\sigma$ curve and the noise level $\sigma_{60}$ where pairwise accuracy falls to 60\%, against $\sigma_{\rm bif}$; (iii) $\sigma_{\rm bif}$ on the held-out test set as a population reference.

In [ ]:
import os; os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import sys, subprocess, pickle, glob, numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math, time, copy
import matplotlib.pyplot as plt
from itertools import combinations
from torchvision import datasets
from scipy.signal import fftconvolve
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print(dev); rng = np.random.default_rng(0); torch.manual_seed(0); plt.rcParams['figure.dpi'] = 120
def load(ds_cls, root):
    d = ds_cls(root, train=True, download=True); return (d.data.numpy().astype(np.float32)/127.5 - 1.0).reshape(-1, 784), d.targets.numpy()
X_MN, Y_MN = load(datasets.MNIST, '/kaggle/working/mnist')

GRID = np.linspace(-4, 4, 4001); DG = GRID[1]-GRID[0]
def smooth_hist(m0, tau, grid=GRID):
    dg = grid[1]-grid[0]; h, _ = np.histogram(m0, bins=np.append(grid-dg/2, grid[-1]+dg/2), density=True); k = np.exp(-grid**2/(2*tau)); k /= k.sum()
    return np.maximum(fftconvolve(h, k, mode='same'), 1e-300)
def _merge(pts, merge=0.02):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts
def stationary_points(p):
    d_ = np.gradient(np.log(p), GRID); keep = p > 1e-4*p.max()
    idx = np.where((np.sign(d_[:-1]) != np.sign(d_[1:])) & keep[:-1] & keep[1:])[0]
    return _merge([(float(GRID[i]-d_[i]*DG/(d_[i+1]-d_[i])) if d_[i+1] != d_[i] else float(GRID[i]), 'max' if d_[i] > 0 else 'min') for i in idx])
def n_max(pts): return sum(k == 'max' for _, k in pts)
def birth_bisect(get_pts, taus=np.geomspace(0.05, 3.0, 60), iters=10):
    ts = sorted(taus, reverse=True); hi = None
    for tau in ts:
        pts = get_pts(tau)
        if n_max(pts) >= 2 and hi is not None: lo = tau; break
        if n_max(pts) == 1: hi = tau
    else: return None
    for _ in range(iters):
        mid = np.sqrt(lo*hi)
        if n_max(get_pts(mid)) >= 2: lo = mid
        else: hi = mid
    pm = [p for p, k in get_pts(hi) if k == 'max']; prev = pm[0] if pm else 0.0; pl = get_pts(lo)
    mx = sorted(p for p, k in pl if k == 'max'); mn = [p for p, k in pl if k == 'min']
    return float(lo), float(abs((mn[0] if mn else 0.5*(mx[0]+mx[-1])) - prev))
def lda_direction(Xa, Xb, lam=1.0):
    mu_a, mu_b = Xa.mean(0), Xb.mean(0); delta = mu_a-mu_b
    u = np.linalg.solve(np.cov(Xa.T)+np.cov(Xb.T)+lam*np.eye(Xa.shape[1]), delta); u /= np.linalg.norm(u)
    return u.astype(np.float32), (0.5*(mu_a+mu_b)).astype(np.float32), 0.5*float(delta @ u)
def balanced(X, Y, pair):
    Xa = X[Y == pair[0]]; Xb = X[Y == pair[1]]; n = min(len(Xa), len(Xb)); return Xa[:n], Xb[:n]

SIGMA_DATA = 0.5
class ResBlock(nn.Module):
    def __init__(self, c, emb):
        super().__init__(); self.n1 = nn.GroupNorm(8, c); self.c1 = nn.Conv2d(c, c, 3, padding=1); self.n2 = nn.GroupNorm(8, c); self.c2 = nn.Conv2d(c, c, 3, padding=1); self.e = nn.Linear(emb, c)
    def forward(self, x, e):
        h = self.c1(F.silu(self.n1(x))) + self.e(e)[:, :, None, None]; return x + self.c2(F.silu(self.n2(h)))
class UNet(nn.Module):
    def __init__(self, ch=64, emb=128):
        super().__init__(); self.emb = nn.Sequential(nn.Linear(64, emb), nn.SiLU(), nn.Linear(emb, emb)); self.inp = nn.Conv2d(1, ch, 3, padding=1)
        self.d1 = ResBlock(ch, emb); self.down = nn.Conv2d(ch, 2*ch, 3, stride=2, padding=1); self.d2 = ResBlock(2*ch, emb); self.mid = ResBlock(2*ch, emb)
        self.up = nn.ConvTranspose2d(2*ch, ch, 4, stride=2, padding=1); self.u1 = ResBlock(ch, emb); self.out = nn.Conv2d(ch, 1, 3, padding=1)
    def forward(self, x, c_noise):
        f = torch.exp(torch.arange(32, device=x.device) * -(math.log(1000)/31)); e = self.emb(torch.cat([torch.sin(c_noise[:, None]*f), torch.cos(c_noise[:, None]*f)], 1))
        h1 = self.d1(self.inp(x), e); h2 = self.mid(self.d2(self.down(h1), e), e); return self.out(self.u1(self.up(h2) + h1, e))
def denoise(net, x, sigma):
    s = sigma.view(-1, 1, 1, 1); c_skip = SIGMA_DATA**2/(s**2 + SIGMA_DATA**2); c_out = s*SIGMA_DATA/torch.sqrt(s**2 + SIGMA_DATA**2)
    return c_skip*x + c_out*net(x/torch.sqrt(s**2 + SIGMA_DATA**2), 0.25*torch.log(sigma))
def train(pair, seed, epochs=20, bs=128, lr=2e-4, P_mean=-0.6, P_std=1.4, ema_decay=0.999):
    torch.manual_seed(seed); X = torch.from_numpy(X_MN[np.isin(Y_MN, pair)]).view(-1, 1, 28, 28).to(dev)
    net = UNet().to(dev); ema = copy.deepcopy(net); opt = torch.optim.Adam(net.parameters(), lr=lr); n = len(X)
    for ep in range(epochs):
        perm = torch.randperm(n, device=dev)
        for i in range(0, n-bs+1, bs):
            x0 = X[perm[i:i+bs]]; sigma = torch.exp(P_mean + P_std*torch.randn(bs, device=dev)); xt = x0 + sigma.view(-1,1,1,1)*torch.randn_like(x0)
            w = (sigma**2 + SIGMA_DATA**2)/(sigma*SIGMA_DATA)**2; loss = (w.view(-1,1,1,1)*(denoise(net, xt, sigma) - x0)**2).mean(); opt.zero_grad(); loss.backward(); opt.step()
            with torch.no_grad():
                for pe, pn in zip(ema.parameters(), net.parameters()): pe.lerp_(pn, 1-ema_decay)
    ema.eval(); torch.save(ema.state_dict(), f'/kaggle/working/unet_{pair[0]}{pair[1]}_s{seed}_ep{epochs}.pt'); return ema
class CNN(nn.Module):
    def __init__(self):
        super().__init__(); self.c1 = nn.Conv2d(1, 32, 3, padding=1); self.c2 = nn.Conv2d(32, 64, 3, padding=1); self.fc = nn.Linear(64*7*7, 2)
    def forward(self, x): x = F.max_pool2d(F.relu(self.c1(x)), 2); x = F.max_pool2d(F.relu(self.c2(x)), 2); return self.fc(x.flatten(1))
def train_cnn(Xa, Xb, epochs=3):
    X = torch.from_numpy(np.concatenate([Xa, Xb])).view(-1, 1, 28, 28).to(dev); Y = torch.cat([torch.zeros(len(Xa)), torch.ones(len(Xb))]).long().to(dev); net = CNN().to(dev); opt = torch.optim.Adam(net.parameters(), 1e-3)
    for _ in range(epochs):
        perm = torch.randperm(len(X), device=dev)
        for i in range(0, len(X), 256): idx = perm[i:i+256]; loss = F.cross_entropy(net(X[idx]), Y[idx]); opt.zero_grad(); loss.backward(); opt.step()
    return net.eval()
S_MAX, S_MIN, RHO = 20.0, 0.002, 7.0; SIG_TAB = np.geomspace(0.05, S_MAX, 48)
@torch.no_grad()
def drift_tables(net, Xflat, u, mid, half, n=20000, nbins=80):
    ut = torch.from_numpy(u).to(dev); midt = torch.from_numpy(mid).to(dev); m0 = (Xflat-mid) @ u / half; tabs = []
    for sigma in SIG_TAB:
        x0 = torch.from_numpy(Xflat[rng.choice(len(Xflat), n)]).to(dev); xt = x0 + sigma*torch.randn_like(x0)
        D = torch.cat([denoise(net, xb.view(-1,1,28,28), torch.full((len(xb),), float(sigma), device=dev)).view(len(xb), -1) for xb in xt.split(2000)])
        m = ((xt-midt) @ ut / half).cpu().numpy(); sm = (half*((D-xt) @ ut)/sigma**2).cpu().numpy()
        lo, hi = np.quantile(m, [0.001, 0.999]); edges = np.linspace(lo, hi, nbins+1); b = np.clip(np.digitize(m, edges)-1, 0, nbins-1)
        cnt = np.bincount(b, minlength=nbins).astype(float); s = np.bincount(b, weights=sm, minlength=nbins); k = np.exp(-0.5*(np.arange(-5, 6)/1.5)**2)
        cen = 0.5*(edges[:-1]+edges[1:]); gnet = np.convolve(s, k, 'same')/np.maximum(np.convolve(cnt, k, 'same'), 1e-9)
        c = np.linspace(min(lo, -30), max(hi, 30), 12001); p = smooth_hist(m0, sigma**2/half**2, grid=c); gex = np.interp(cen, c, np.gradient(np.log(p), c)); tabs.append((cen, gex - gnet))
    return tabs
def band_corr(tabs, half, lo_tau, hi_tau):
    def corr(m_np, sigma):
        tau = sigma**2/half**2
        if not (lo_tau <= tau <= hi_tau): return np.zeros_like(m_np)
        k = int(np.argmin(np.abs(np.log(SIG_TAB) - np.log(sigma)))); cen, dg = tabs[k]; return np.interp(m_np, cen, dg)
    return corr
@torch.no_grad()
def sample(net, u, mid, half, corr=None, steps=40, churn=0.0, n=4000, track=False):
    ut = torch.from_numpy(u).to(dev); midt = torch.from_numpy(mid).to(dev); ut4 = ut.view(1, 1, 28, 28)
    def Dc(x, s):
        D = denoise(net, x, s.expand(n))
        if corr is None: return D
        m = ((x.view(n, -1) - midt) @ ut / half).cpu().numpy(); dg = torch.from_numpy(corr(m, float(s)).astype(np.float32)).to(dev); return D + (float(s)**2/half)*dg.view(n, 1, 1, 1)*ut4
    i = torch.arange(steps, device=dev); sig = torch.cat([(S_MAX**(1/RHO) + i/(steps-1)*(S_MIN**(1/RHO)-S_MAX**(1/RHO)))**RHO, torch.zeros(1, device=dev)]); x = torch.randn(n, 1, 28, 28, device=dev)*sig[0]; mh = []
    for k in range(steps):
        s0, s1 = sig[k], sig[k+1]
        if churn > 0 and 0.01 <= s0 <= S_MAX: gam = min(churn/steps, 2**0.5-1); sh = s0*(1+gam); x = x + torch.sqrt(sh**2 - s0**2)*torch.randn_like(x); s0 = sh
        D0 = Dc(x, s0)
        if track: mh.append(((D0.view(n, -1) - midt) @ ut / half).cpu().numpy())
        d0 = (x - D0)/s0; x1 = x + (s1-s0)*d0
        if s1 > 0: d1 = (x1 - Dc(x1, s1))/s1; x = x + (s1-s0)*0.5*(d0+d1)
        else: x = x1
    return (x, sig[:-1].cpu().numpy(), np.array(mh)) if track else x

## A. Band-limited drift replacement: which noise band carries the bias?

In [ ]:
PAIRS = [(0, 1), (3, 8), (0, 8), (2, 7)]; SEEDS = [0, 1, 2]
BANDS = {'none': None, 'high (tau>1.5)': (1.5, 1e9), 'window (0.3-1.5)': (0.3, 1.5), 'low (tau<0.3)': (0.0, 0.3), 'all sigma': (0.0, 1e9)}
RES = {}; NETS = {}; INFO = {}
for pair in PAIRS:
    Xa, Xb = balanced(X_MN, Y_MN, pair); u, mid, half = lda_direction(Xa, Xb); Xflat = np.concatenate([Xa, Xb]); cnn = train_cnn(Xa, Xb)
    base = float((cnn(torch.from_numpy(Xflat).view(-1,1,28,28).to(dev)).argmax(1) == 0).float().mean()); INFO[pair] = (u, mid, half, cnn, base)
    for seed in SEEDS:
        t0 = time.time(); net = train(pair, seed); NETS[(pair, seed)] = net; tabs = drift_tables(net, Xflat, u, mid, half)
        for bname, band in BANDS.items():
            corr = None if band is None else band_corr(tabs, half, *band)
            for sname, kw in [('heun40', dict(steps=40)), ('sde40', dict(steps=40, churn=40))]:
                x = sample(net, u, mid, half, corr, **kw); RES[(pair, seed, bname, sname)] = float((cnn(x).argmax(1) == 0).float().mean()) - base
        print(f"  {pair} seed {seed} done ({time.time()-t0:.0f}s)")
print(f"\nExcess '+' fraction over real-data baseline, mean +/- sd over 3 seeds (n=4000 per cell, s.e. ~0.008):")
print(f"{'pair':>6} {'sampler':>7} " + ' '.join(f"{b:>18}" for b in BANDS))
for pair in PAIRS:
    for sname in ['heun40', 'sde40']:
        vals = [np.array([RES[(pair, s, b, sname)] for s in SEEDS]) for b in BANDS]
        print(f"{pair!s:>6} {sname:>7} " + ' '.join(f"{v.mean():+8.3f}+/-{v.std():5.3f}" for v in vals))

## B. Slow passage: where do trajectories commit relative to the transition?

In [ ]:
def commit_tau(sigs, mh, half):
    """Per trajectory: the noise level (as tau) of the LAST sign change of the denoised LDA coordinate. Trajectories that never flip commit 'at the start'."""
    sgn = np.sign(mh); flips = (sgn[1:] != sgn[:-1]); last = np.array([np.max(np.where(f)[0])+1 if f.any() else 0 for f in flips.T]); return sigs[last]**2/half**2
def net_bisect(net, Xflat, u, mid, half):
    tabs = drift_tables(net, Xflat, u, mid, half, n=15000)
    def pts(tau):
        k = int(np.argmin(np.abs(np.log(SIG_TAB) - np.log(half*np.sqrt(tau))))); cen, dg = tabs[k]
        m0 = (Xflat-mid) @ u / half; c = np.linspace(-30, 30, 12001); p = smooth_hist(m0, tau, grid=c); gex = np.interp(cen, c, np.gradient(np.log(p), c)); gnet = gex - dg
        out = []
        for i in range(len(cen)-1):
            if np.sign(gnet[i]) != np.sign(gnet[i+1]) and gnet[i] != 0: out.append((float(cen[i]-gnet[i]*(cen[i+1]-cen[i])/(gnet[i+1]-gnet[i])), 'max' if gnet[i] > 0 else 'min'))
        return _merge(out, merge=0.05)
    return birth_bisect(pts, taus=np.geomspace(0.1, 3.0, 30), iters=6)
print(f"{'pair':>6} {'exact tau_bif':>14} {'net tau_bif (s0)':>17} {'commit tau: median [IQR], Heun-40':>36} {'SDE-40':>22} {'Heun-160':>22}")
for pair in PAIRS:
    u, mid, half, cnn, base = INFO[pair]; Xa, Xb = balanced(X_MN, Y_MN, pair); Xflat = np.concatenate([Xa, Xb]); m0 = (Xflat-mid) @ u / half
    ex = birth_bisect(lambda t: stationary_points(smooth_hist(m0, t))); nb_ = net_bisect(NETS[(pair, 0)], Xflat, u, mid, half)
    row = []
    for sname, kw in [('heun40', dict(steps=40)), ('sde40', dict(steps=40, churn=40)), ('heun160', dict(steps=160))]:
        x, sigs, mh = sample(NETS[(pair, 0)], u, mid, half, None, track=True, n=2000, **kw); ct = commit_tau(sigs, mh, half); q = np.percentile(ct, [25, 50, 75]); row.append(f"{q[1]:.2f} [{q[0]:.2f}, {q[2]:.2f}]")
    print(f"{pair!s:>6} {ex[0]:14.3f} {(nb_[0] if nb_ else float('nan')):17.3f} {row[0]:>36} {row[1]:>22} {row[2]:>22}")
# exact-score reference: d=784 mixture with width asymmetry along the class axis (as in Step 17)
def make_mix(d=784, wa=0.15, wb=0.45, iso=0.3, seed=0):
    r = np.random.default_rng(seed); s = np.sign(r.standard_normal(d)).astype(np.float64); uh = s/np.sqrt(d)
    def score(x, sigma):
        a = iso**2 + sigma**2; W = []; G = []
        for mu, wc in [(s, wa), (-s, wb)]:
            b = wc**2*d; dx = x - mu; proj = dx @ uh; W.append(-0.5*(((dx**2).sum(1) - (b/(a+b))*proj**2)/a) - 0.5*((d-1)*np.log(a) + np.log(a+b))); G.append(-(dx - (b/(a+b))*proj[:, None]*uh[None, :])/a)
        m = np.maximum(W[0], W[1]); pa = np.exp(W[0]-m); pb = np.exp(W[1]-m); wgt = (pa/(pa+pb))[:, None]; return wgt*G[0] + (1-wgt)*G[1]
    return s, score
s, score = make_mix(); d = 784; r = np.random.default_rng(1); n = 2000; steps = 160
i = np.arange(steps); sig = np.append((S_MAX**(1/RHO) + i/(steps-1)*(S_MIN**(1/RHO)-S_MAX**(1/RHO)))**RHO, 0.0); x = r.standard_normal((n, d))*sig[0]; mh = []
half_x = float(np.sqrt(d))          # class means at +/-s, so u = s/sqrt(d), half = sqrt(d)
for k in range(steps):
    s0, s1 = sig[k], sig[k+1]; sc = score(x, s0); D0 = x + s0**2*sc; mh.append((D0 @ s)/d); d0 = -s0*sc; x1 = x + (s1-s0)*d0
    if s1 > 0: d1 = -s1*score(x1, s1); x = x + (s1-s0)*0.5*(d0+d1)
    else: x = x1
mh = np.array(mh); ct = commit_tau(sig[:-1], mh, half_x); A = s + 0.15*r.standard_normal((6000, 1))*s + 0.3*r.standard_normal((6000, d)); B = -s + 0.45*r.standard_normal((6000, 1))*s + 0.3*r.standard_normal((6000, d))
u, mid, half = lda_direction(A.astype(np.float32), B.astype(np.float32)); m0 = np.concatenate([(A-mid) @ u, (B-mid) @ u])/half; ex = birth_bisect(lambda t: stationary_points(smooth_hist(m0, t)))
q = np.percentile(ct*(half_x**2/half**2), [25, 50, 75])
print(f"exact-score mixture (width ratio 3): exact tau_bif {ex[0]:.3f}; commit tau (Heun-160) median {q[1]:.2f} [IQR {q[0]:.2f}, {q[2]:.2f}]  -> lag = {ex[0]/q[1]:.2f}x in tau (commitment after the transition if >1)")

## C. CIFAR-10: floor, classifier blindness, and a test-set reference

In [ ]:
if not os.path.exists('/kaggle/working/edm'): subprocess.run(['git', 'clone', '-q', 'https://github.com/NVlabs/edm.git', '/kaggle/working/edm'], check=True)
ds = datasets.CIFAR10('/kaggle/working/cifar', train=True, download=True); dst = datasets.CIFAR10('/kaggle/working/cifar', train=False, download=True)
X = torch.from_numpy((np.asarray(ds.data).astype(np.float32)/127.5 - 1.0).transpose(0, 3, 1, 2)); Y = np.asarray(ds.targets)
XT = torch.from_numpy((np.asarray(dst.data).astype(np.float32)/127.5 - 1.0).transpose(0, 3, 1, 2)); YT = np.asarray(dst.targets)
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
class FiLMBlock(nn.Module):
    def __init__(self, cin, cout, emb, stride=1):
        super().__init__(); self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout); self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False); self.b2 = nn.BatchNorm2d(cout)
        self.film = nn.Linear(emb, 2*cout); self.sc = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout)) if (stride != 1 or cin != cout) else nn.Identity()
    def forward(self, x, e):
        g, b = self.film(e).chunk(2, 1); h = F.silu(self.b1(self.c1(x))); h = self.b2(self.c2(h))*(1+g[:, :, None, None]) + b[:, :, None, None]; return F.silu(h + self.sc(x))
class NoisyResNet(nn.Module):
    def __init__(self, ch=64, emb=128):
        super().__init__(); self.emb = nn.Sequential(nn.Linear(32, emb), nn.SiLU(), nn.Linear(emb, emb)); self.inp = nn.Conv2d(3, ch, 3, 1, 1)
        self.blocks = nn.ModuleList([FiLMBlock(ch, ch, emb), FiLMBlock(ch, 2*ch, emb, 2), FiLMBlock(2*ch, 2*ch, emb), FiLMBlock(2*ch, 4*ch, emb, 2), FiLMBlock(4*ch, 4*ch, emb)]); self.fc = nn.Linear(4*ch, 10)
    def forward(self, x, sigma):
        f = torch.exp(torch.arange(16, device=x.device)*-(math.log(100)/15)); t = 0.25*torch.log(sigma)[:, None]*f; e = self.emb(torch.cat([torch.sin(t), torch.cos(t)], 1)); h = self.inp(x)
        for b in self.blocks: h = b(h, e)
        return self.fc(h.mean((2, 3)))
def train_clf10(seed=0, epochs=25, bs=256, s_lo=0.05, s_hi=10.0):
    torch.manual_seed(seed); Xd = X.to(dev); Yd = torch.from_numpy(Y).long().to(dev); clf = NoisyResNet().to(dev); opt = torch.optim.AdamW(clf.parameters(), 2e-3, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=epochs*(len(Xd)//bs)); t0 = time.time()
    for ep in range(epochs):
        clf.train(); perm = torch.randperm(len(Xd), device=dev)
        for i in range(0, len(Xd)-bs+1, bs):
            j = perm[i:i+bs]; x0 = Xd[j]
            if torch.rand(1).item() < 0.5: x0 = x0.flip(3)
            sigma = torch.exp(torch.empty(bs, device=dev).uniform_(math.log(s_lo), math.log(s_hi))); xt = x0 + sigma.view(-1, 1, 1, 1)*torch.randn_like(x0)
            loss = F.cross_entropy(clf(xt, sigma), Yd[j]); opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    clf.eval(); torch.save(clf.state_dict(), f'/kaggle/working/noisy_resnet_seed{seed}.pt'); print(f"classifier trained ({time.time()-t0:.0f}s)"); return clf
found = glob.glob('/kaggle/input/**/noisy_resnet_seed0.pt', recursive=True)
if found: CLF10 = NoisyResNet().to(dev); CLF10.load_state_dict(torch.load(found[0], map_location=dev)); CLF10.eval(); print('loaded classifier')
else: CLF10 = train_clf10(0)
@torch.no_grad()
def logits_batched(x, sigma, bs=1000): return torch.cat([CLF10(xb, torch.full((len(xb),), float(sigma), device=dev)) for xb in x.split(bs)])

GRIDC = np.linspace(-4.0, 4.0, 4001); DGC = GRIDC[1]-GRIDC[0]
def ref_score(m):
    m = np.clip(m, -8, 8); bw = max(1.5*1.06*m.std()*len(m)**(-1/5), 1e-3)
    h, _ = np.histogram(m, bins=np.append(GRIDC-DGC/2, GRIDC[-1]+DGC/2), density=True); k = np.exp(-GRIDC**2/(2*bw**2)); k /= k.sum(); p = np.maximum(fftconvolve(h, k, 'same'), 1e-300)
    return np.gradient(np.log(p), GRIDC), p
def stationary_points_c(sm, p, lo=-3.5, hi=3.5, prominence=0.05):
    keep = (GRIDC > lo) & (GRIDC < hi) & (p > 1e-3*p.max()); idx = np.where((np.sign(sm[:-1]) != np.sign(sm[1:])) & keep[:-1] & keep[1:])[0]
    pts = _merge([(float(GRIDC[i]-sm[i]*DGC/(sm[i+1]-sm[i])) if sm[i+1] != sm[i] else float(GRIDC[i]), 'max' if sm[i] > 0 else 'min') for i in idx], merge=0.03)
    logp = np.concatenate([[0.0], np.cumsum(0.5*(sm[1:]+sm[:-1])*DGC)]); lp = lambda m: float(np.interp(m, GRIDC, logp)); ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if pts[a][1] != pts[a+1][1] and abs(lp(pts[a][0]) - lp(pts[a+1][0])) < prominence: del pts[a:a+2]; ch = True; break
    return pts
def cusp_fit_c(m, f, w, window=1.0, b_min=0.02, s_max=0.6):
    ok = (np.abs(m) < window) & (w > 0); c3, c2, c1, c0 = np.polyfit(m[ok], f[ok], 3, w=np.sqrt(w[ok])); b = max(-c3, b_min); c3 = -b; s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    return dict(h=c0 + c1*s + c2*s**2 + c3*s**3, a=c1 + 3*c3*s**2 + 2*c2*s, b=b, clamped=(-c3 <= b_min+1e-12))
def bisect_sigma(get_pts, sigmas, iters=6):
    ss = sorted(sigmas, reverse=True); hi = None
    for s_ in ss:
        pts = get_pts(s_)
        if n_max(pts) >= 2 and hi is not None: lo = s_; break
        if n_max(pts) == 1: hi = s_
    else: return None
    for _ in range(iters):
        mid = np.sqrt(lo*hi)
        if n_max(get_pts(mid)) >= 2: lo = mid
        else: hi = mid
    pm = [p for p, k in get_pts(hi) if k == 'max']; prev = pm[0] if pm else 0.0; pl = get_pts(lo); mx = sorted(p for p, k in pl if k == 'max'); mn = [p for p, k in pl if k == 'min']
    return float(lo), float(abs((mn[0] if mn else 0.5*(mx[0]+mx[-1])) - prev))
SIGMAS = np.geomspace(0.1, 30.0, 30)
def pair_logit_samples(pair, Xsrc, Ysrc, sigma, n=20000):
    idx = np.where(np.isin(Ysrc, pair))[0]; Xp = Xsrc[idx].to(dev); Yp = (Ysrc[idx] == pair[0])
    j = np.random.default_rng(int(sigma*1000) % 2**31).integers(len(Xp), size=n); xt = Xp[j] + sigma*torch.randn(len(j), 3, 32, 32, device=dev)
    l = logits_batched(xt, sigma); z = (l[:, pair[0]] - l[:, pair[1]]).cpu().numpy(); ya = Yp[j]; return z, ya
def census_pair(pair, Xsrc, Ysrc, symmetrise=False, min_sep=0.3):
    cache = {}
    def ref_at(sigma):
        key = round(float(sigma), 5)
        if key not in cache:
            z, ya = pair_logit_samples(pair, Xsrc, Ysrc, sigma); mu_a, mu_b = z[ya].mean(), z[~ya].mean(); half = 0.5*(mu_a-mu_b); mid = 0.5*(mu_a+mu_b); sep = abs(half)/(0.5*(z[ya].std()+z[~ya].std()))
            if sep < min_sep: cache[key] = (None, None, sep); return cache[key]
            if symmetrise:                                   # class b replaced by the mirror image of an independent half of class a's logits
                za = z[ya]; idx = np.random.default_rng(int(sigma*777) % 2**31).permutation(len(za)); k = len(za)//2; m = np.concatenate([(za[idx[:k]]-mid)/half, -(za[idx[k:2*k]]-mid)/half])
            else: m = (z - mid)/half
            sm, p = ref_score(m); cache[key] = (sm, p, sep)
        return cache[key]
    def pts_at(sigma):
        sm, p, sep = ref_at(sigma); return [(0.0, 'max')] if sm is None else stationary_points_c(sm, p)
    bj = bisect_sigma(pts_at, SIGMAS)
    if bj is None: return None
    sm, p, sep = ref_at(bj[0]); f = cusp_fit_c(GRIDC, sm, p); return dict(sigma_bif=bj[0], jump=bj[1], sep=sep, **f)

print("== (i) symmetrised-control floor along the log-odds (train set) ==")
fl = []
for pair in combinations(range(10), 2):
    for rep in range(2):
        r = census_pair(pair, X, Y, symmetrise=True)
        if r: fl.append(abs(r['h']))
FLOOR_C = float(np.percentile(fl, 95)); print(f"{len(fl)} controls; |h| median {np.median(fl):.4f}, 95th pct {FLOOR_C:.4f}, max {max(fl):.4f}")

print("\n== (ii) census (train), classifier blindness sigma_60, and (iii) census on the TEST set ==")
print(f"{'pair':>22} {'sigma_bif train':>16} {'sigma_bif test':>15} {'sigma_60':>9} {'bif/blind':>10} {'h train':>9} {'above floor':>12}")
rows = []
for pair in combinations(range(10), 2):
    rt = census_pair(pair, X, Y); rs = census_pair(pair, XT, YT)
    accs = []
    for sg in np.geomspace(0.1, 30, 25):
        z, ya = pair_logit_samples(pair, X, Y, sg, n=4000); accs.append(float(((z > 0) == ya).mean()))
    accs = np.array(accs); sgs = np.geomspace(0.1, 30, 25); s60 = float(np.interp(0.60, accs[::-1], sgs[::-1])) if accs.min() < 0.6 < accs.max() else float('nan')
    if rt is None: print(f"{CLASSES[pair[0]]+'/'+CLASSES[pair[1]]:>22}  never bimodal (train)   sigma_60={s60:.2f}"); continue
    rows.append(dict(pair=pair, s_train=rt['sigma_bif'], s_test=(rs['sigma_bif'] if rs else np.nan), s60=s60, h=rt['h'], h_test=(rs['h'] if rs else np.nan)))
    print(f"{CLASSES[pair[0]]+'/'+CLASSES[pair[1]]:>22} {rt['sigma_bif']:16.3f} {(rs['sigma_bif'] if rs else float('nan')):15.3f} {s60:9.2f} {rt['sigma_bif']/s60:10.2f} {rt['h']:+9.4f} {str(abs(rt['h']) > FLOOR_C):>12}")
st = np.array([r['s_train'] for r in rows]); se = np.array([r['s_test'] for r in rows]); s60 = np.array([r['s60'] for r in rows]); H = np.array([r['h'] for r in rows]); HT = np.array([r['h_test'] for r in rows])
ok = np.isfinite(se) & np.isfinite(s60)
print(f"\npairs above the CIFAR floor ({FLOOR_C:.4f}): {(np.abs(H) > FLOOR_C).sum()}/{len(rows)}")
print(f"train vs test: corr(log sigma_bif) = {np.corrcoef(np.log(st[ok]), np.log(se[ok]))[0,1]:.3f}, median ratio test/train {np.nanmedian(se/st):.2f}; corr(h) = {np.corrcoef(H[np.isfinite(HT)], HT[np.isfinite(HT)])[0,1]:.3f}")
print(f"blindness: median sigma_bif/sigma_60 = {np.nanmedian(st/s60):.2f} (range {np.nanmin(st/s60):.2f}-{np.nanmax(st/s60):.2f}); corr(log sigma_bif, log sigma_60) = {np.corrcoef(np.log(st[ok]), np.log(s60[ok]))[0,1]:.3f}")
VEH = {0, 1, 8, 9}; kind = np.array(['vv' if (p['pair'][0] in VEH and p['pair'][1] in VEH) else 'aa' if (p['pair'][0] not in VEH and p['pair'][1] not in VEH) else 'va' for p in rows])
for k in ['va', 'vv', 'aa']: print(f"  {k}: median sigma_bif {np.median(st[kind == k]):.2f}, median sigma_60 {np.nanmedian(s60[kind == k]):.2f}, median ratio {np.nanmedian((st/s60)[kind == k]):.2f}")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].scatter(s60, st, s=16, c=['C0' if k == 'va' else 'C1' if k == 'vv' else 'C2' for k in kind]); lim = [0.2, 40]; ax[0].plot(lim, lim, 'k--', lw=0.7); ax[0].set_xscale('log'); ax[0].set_yscale('log'); ax[0].set_xlabel('sigma_60 (classifier blindness)'); ax[0].set_ylabel('sigma_bif (train)'); ax[0].set_title('is the hierarchy just where the classifier goes blind?')
ax[1].scatter(st, se, s=16); ax[1].plot(lim, lim, 'k--', lw=0.7); ax[1].set_xscale('log'); ax[1].set_yscale('log'); ax[1].set_xlabel('sigma_bif (train)'); ax[1].set_ylabel('sigma_bif (test)'); ax[1].set_title('train vs held-out reference')
plt.tight_layout(); plt.show()

## What to report back
**A.** The band table: which band's replacement removes the bias (mean ± sd over seeds), per pair and sampler.
**B.** The commitment table: median commit τ vs exact and network τ_bif, and the exact-score reference lag.
**C.** The CIFAR floor and the count above it; train-vs-test correlations; the blindness ratio and its per-category medians; both scatter plots.